In [48]:
import pandas as pd
from datetime import datetime, time
from Strategies.Sparse_momentum.ob_attributes import TR_attributes
from Database.TPData import TPData
from Database.DB_reader import Database

In [49]:
def process_db(data_dict, trd_dict):
    out_dict = {}
    for m, df_data in data_dict.items():
        df_trd_all = trd_dict[m]
        df_trd_aux = df_trd_all.loc[df_data.index, 'tradeid']
        out_dict[m] = pd.concat([df_trd_aux, df_data], axis=1).sort_index()
    return out_dict

# Selecting all trades since 2024

In [50]:
# 1. Read data from source DB
conn = Database('timescaledb')

query=f"""select distinct datetime, nanotime, tradeid from  public.trades 
          where datetime>='2024-12-01' and datetime<='2025-05-31' 
          and EXTRACT(HOUR FROM datetime) BETWEEN 8 AND 18
          and instid in ('10641710', '10001075', '10100480', '10012528', '10002806')
          order by datetime asc, nanotime asc""" # where rownum <= 100"""
df=conn.execute(query)


print(f"✅ Loaded {len(df)} rows from source database.")

Connected to the database timescaledb
Disconnected from the database timescaledb
✅ Loaded 2429390 rows from source database.


In [51]:
contract='dey1'

In [52]:
data_class = TPData()
data_class.create_connection('OracleSQL')

mkt_list = [contract[0:2]]
tenor_list = [contract[2]]
tn_list = [int(contract[-1])]

prod = 'base'
venue_list = ['eex']
start_date = datetime(2024, 12, 1)
end_date = datetime(2025, 5, 31)

allwd_broker_ids = [1441]

sample_dates = pd.date_range(start_date, end_date, freq='B')

n_s = 2

dates = pd.date_range(start_date, end_date, freq='B')
product_date = [dates.shift(1, freq='B') if t == 'da' else
                dates.shift(1, freq='D') if t == 'd' else
                dates.shift(tn, freq='W-MON') if t == 'w' else
                (dates + n_s * dates.freq).shift(tn, freq='2QS-Apr') if t in ['sum', 'win'] else
                (dates + n_s * dates.freq).shift(tn, freq='YS') if t in ['dec'] else
                (dates + n_s * dates.freq).shift(tn, freq=t.upper() + 'S')
                for t, tn in zip(tenor_list, tn_list)]

start_time = time(8, 30, 0)
end_time = time(17, 30, 0)

tr_data_dict = {m + t + str(n): [] for m, t, n in zip(mkt_list, tenor_list, tn_list)}
agg_dict = {'price': 'sum', 'volume': 'sum', 'action': 'median',
            'broker_id': 'median', 'count': 'sum', 'tradeid': 'first'}

for m, t, n, p_dates in zip(mkt_list, tenor_list, tn_list, product_date):
    df_tr = pd.DataFrame([])
    series = pd.Series(p_dates, index=dates)
    for p_d, ds in series.groupby(series).groups.items():
        bT = datetime.combine(ds[0], start_time)
        eT = datetime.combine(ds[-1], end_time)
        # Trades
        df_tr_aux = data_class.get_trades(m, t, venue_list, p_d, bT, eT, prod)
        # Filter by broker
        if not allwd_broker_ids or t == 'da':
            pass
        else:
            df_tr_aux = df_tr_aux[df_tr_aux['broker_id'].isin(allwd_broker_ids)]
        try:
            df_tr_aux = df_tr_aux.between_time(start_time, end_time)
        except(TypeError):
            pass
        # Group trades
        df_tr_aux['count'] = 1
        df_tr_aux['price'] *= df_tr_aux['volume']
        df_tr_aux = df_tr_aux.groupby(df_tr_aux.index).agg(agg_dict)
        df_tr_aux['price'] /= df_tr_aux['volume']
        df_tr = pd.concat([df_tr, df_tr_aux])
        del df_tr_aux
    tr_data_dict[m + t + str(n)] = df_tr

data_p_ = pd.concat({k: v['price'] for k, v in tr_data_dict.items()}, axis=1)
data_v_ = pd.concat({k: v['volume'] for k, v in tr_data_dict.items()}, axis=1)

trAtt = TR_attributes([])
# Market
mkt_code_list = [m + t + str(tn) for m, t, tn in zip(mkt_list, tenor_list, tn_list)]

### VPIN ###
vol_bucket_dict = {'dem1': 20, 'dem2': 13, 'deq1': 12, 'dey1': 13}
bucket_n_list = [10, 20, 50, 100]
n_days = 3

vpin_dict = {}
for m in mkt_code_list:
    df_data_vpin = pd.concat([data_p_[m], data_v_[m]], axis=1).dropna()
    df_data_vpin.columns = ['price', 'volume']
    df_data_vpin.index.name = 'timestamp'
     
    df_vpin = pd.DataFrame()
    for n_buckets in bucket_n_list:
        df_aux = trAtt.vpin(df_data_vpin, vol_bucket_dict[m], n_buckets, n_days=n_days)
        df_aux.columns = ['vpin_' + str(n_buckets)+f'_{contract}']
        if df_vpin.empty:
            df_vpin = df_aux
        else:
            df_vpin = pd.concat([df_vpin, df_aux], axis=1)
    vpin_dict[m] = df_vpin

db_dict = process_db(vpin_dict, tr_data_dict)

Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle


In [53]:
db_dict[contract].reset_index()

,timestamp,tradeid,vpin_10_dey1,vpin_20_dey1,vpin_50_dey1,vpin_100_dey1
0,2024-12-02 08:37:50.576868536,Eurex T7/DEBY012025-20241202/295/2,NaN,NaN,NaN,NaN
1,2024-12-02 08:46:14.214299182,Eurex T7/DEBY012025-20241202/421/1,NaN,NaN,NaN,NaN
2,2024-12-02 09:03:57.512967777,Eurex T7/DEBY012025-20241202/598/1,NaN,NaN,NaN,NaN
3,2024-12-02 09:10:00.215727114,Eurex T7/DEBY012025-20241202/698/1,NaN,NaN,NaN,NaN
4,2024-12-02 09:15:26.668768245,Eurex T7/DEBY012025-20241202/767/1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
4875,2025-05-30 15:49:55.371321719,Eurex T7/DEBY012026-20250530/6481/1,0.402290,0.407934,0.438444,0.462826
4876,2025-05-30 16:06:14.520399853,Eurex T7/DEBY012026-20250530/6745/1,0.412417,0.416364,0.421849,0.455442
4877,2025-05-30 16:28:54.717967215,Eurex T7/DEBY012026-20250530/7022/1,0.355957,0.414099,0.427116,0.458008
4878,2025-05-30 16:53:53.455290716,Eurex T7/DEBY012026-20250530/7391/1,0.436856,0.418997,0.429637,0.458281


# Saving into TimescaleDB


In [54]:
vpin_dict = {
    'vpin_10_dem1': 246,
    'vpin_20_dem1': 247,
    'vpin_50_dem1': 248,
    'vpin_100_dem1': 249,
    'vpin_10_dem2': 250,
    'vpin_20_dem2': 251,
    'vpin_50_dem2': 252,
    'vpin_100_dem2': 253,
    'vpin_10_deq1': 254,
    'vpin_20_deq1': 255,
    'vpin_50_deq1': 256,
    'vpin_100_deq1': 257,
    'vpin_10_dey1': 258,
    'vpin_20_dey1': 259,
    'vpin_50_dey1': 260,
    'vpin_100_dey1': 261,
}

In [55]:
df_preds=db_dict[contract].reset_index()

In [56]:
# Step 1: Convert wide df to long format using melt

df_merged = df.merge(
df_preds[['tradeid']+['vpin_' + str(n_buckets)+f'_{contract}' for n_buckets in bucket_n_list]],
on='tradeid', how='left'
)
df_long = df_merged.melt(
    id_vars=['datetime', 'nanotime', 'tradeid'],  # Keep these columns fixed
    value_vars=['vpin_' + str(n_buckets)+f'_{contract}' for n_buckets in bucket_n_list],
    var_name='pred_name',
    value_name='pred_value'
)    
# Step 2: Add pred_name and additional columns if required
df_long['pred_id'] = df_long['pred_name'].map(vpin_dict) # or use a mapping dict if needed
df_long['additional'] = None  # set accordingly if you have additional data

# Step 3: Sort by datetime for proper forward filling
df_long = df_long.sort_values(by=['datetime', 'nanotime','pred_id'], ascending=[True, True, True])

# Step 4: Forward fill within each day separately
#df_long['date_only'] = df_long['datetime'].dt.date
#df_long['pred_value'] = df_long.groupby(['pred_id', 'date_only'])['pred_value'].ffill()

# Optionally drop rows where pred_value is still NaN after forward fill
df_long = df_long.dropna(subset=['pred_value'])

# Step 5: Drop helper columns
#df_long = df_long.drop(columns=['date_only'])

# Locate rows where pred_value is missing and set additional message
mask_missing = df_long['pred_value'].isna()
df_long.loc[mask_missing, 'additional'] = 'Not enough observations to calculate the pred_value!'


# Optional step: enforce data types explicitly
df_long['pred_value'] = df_long['pred_value'].astype(float)

df_long=df_long[['datetime', 'nanotime', 'tradeid', 'pred_id','pred_value']].reset_index(drop=True)


# 2. Connect to TimescaleDB
batch_size=100_000
conn = Database('timescaledb')
conn._connect()

# 3. Insert in batches
total_rows = len(df_long)
for start in range(0, total_rows, batch_size):
    end = min(start + batch_size, total_rows)
    batch = df_long.iloc[start:end]

    batch.to_sql('experimental_dataset_entries', conn.engine,schema='public', index=False, if_exists='append',method='multi')
    print(f"✅ Inserted rows {start} to {end} into TimescaleDB.")

print("🎉 All batches inserted successfully.")

Connected to the database timescaledb
✅ Inserted rows 0 to 19244 into TimescaleDB.
🎉 All batches inserted successfully.
